In [ ]:
import os
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms

DATA_DIR = "data"
BATCH_SIZE = 32
IMAGE_SIZE = 224

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

# Geometric only: small resize, small rotation, small crop.
# No color jitter and no other color change.
train_transform = transforms.Compose(
    [
        transforms.Resize(248),
        transforms.RandomAffine(
            degrees=8,
            scale=(0.94, 1.06),
            interpolation=transforms.InterpolationMode.BILINEAR,
        ),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        normalize,
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize(248),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        normalize,
    ]
)


class NumericImageFolder(datasets.ImageFolder):
    """ImageFolder that uses the numeric folder name as the class index."""

    def find_classes(self, directory):
        classes = [
            entry.name
            for entry in os.scandir(directory)
            if entry.is_dir() and entry.name.isdigit()
        ]
        classes.sort(key=int)
        class_to_idx = {name: int(name) for name in classes}
        return classes, class_to_idx


class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, index):
        image, label = self.subset[index]
        return self.transform(image), label


def painter_names(root):
    names = {}
    for class_dir in Path(root).iterdir():
        if not class_dir.is_dir() or not class_dir.name.isdigit():
            continue
        sample = next(
            path
            for path in class_dir.iterdir()
            if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
        )
        names[int(class_dir.name)] = sample.stem.rsplit("_", 1)[0]
    return dict(sorted(names.items()))


base_dataset = NumericImageFolder(DATA_DIR)
val_size = int(0.2 * len(base_dataset))
train_size = len(base_dataset) - val_size
train_base, val_base = random_split(
    base_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)
train_set = TransformSubset(train_base, train_transform)
val_set = TransformSubset(val_base, val_transform)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

num_classes = len(base_dataset.classes)
painters = painter_names(DATA_DIR)
print(f"train: {len(train_set)}   validation: {len(val_set)}   classes: {num_classes}")
print(painters)


In [ ]:
import copy
import time

import matplotlib.pyplot as plt
from torch import nn
from torchvision import models

CHECKPOINT = "models/resnet152_retrained.pth"
OUTPUT_PATH = "models/resnet152_progressive.pth"

MAX_EPOCHS = 50
INITIAL_LR = 1e-3
LR_PATIENCE = 7
EARLY_STOP_PATIENCE = 14

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

model = models.resnet152(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(
    torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
)

# Classifier first, then each block back to the stem.
LAYER_STAGES = [
    ("fc", [model.fc]),
    ("layer4", [model.layer4]),
    ("layer3", [model.layer3]),
    ("layer2", [model.layer2]),
    ("layer1", [model.layer1]),
    ("stem", [model.conv1, model.bn1]),
]


def trainable_parameters(net):
    return [param for param in net.parameters() if param.requires_grad]


def apply_train_mode(net, trainable_modules):
    # Keep frozen blocks in eval so their batch-norm stats stay fixed.
    net.eval()
    for module in trainable_modules:
        module.train()


def run_epoch(net, loader, trainable_modules, optimizer=None):
    training = optimizer is not None
    if training:
        apply_train_mode(net, trainable_modules)
    else:
        net.eval()

    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = net(images)
            loss = torch.nn.functional.cross_entropy(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


def train_stage(net, stage_name, trainable_modules):
    net.to(device)
    optimizer = torch.optim.Adam(trainable_parameters(net), lr=INITIAL_LR)
    _, baseline_acc = run_epoch(net, val_loader, trainable_modules)
    best_acc = baseline_acc
    best_state = copy.deepcopy(net.state_dict())
    epochs_without_improvement = 0
    print(
        f"{stage_name}  baseline val acc {baseline_acc:.3f}  lr {INITIAL_LR:.1e}",
        flush=True,
    )

    for epoch in range(1, MAX_EPOCHS + 1):
        started = time.time()
        _, train_acc = run_epoch(net, train_loader, trainable_modules, optimizer)
        _, val_acc = run_epoch(net, val_loader, trainable_modules)
        lr = optimizer.param_groups[0]["lr"]
        print(
            f"{stage_name}  epoch {epoch:02d}/{MAX_EPOCHS}  lr {lr:.1e}  "
            f"train acc {train_acc:.3f}  val acc {val_acc:.3f}  "
            f"({time.time() - started:.0f}s)",
            flush=True,
        )

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(net.state_dict())
            epochs_without_improvement = 0
            continue

        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"{stage_name}  stopping at epoch {epoch}: "
                f"{epochs_without_improvement} epochs without improvement",
                flush=True,
            )
            break
        if epochs_without_improvement % LR_PATIENCE == 0:
            optimizer.param_groups[0]["lr"] *= 0.1
            print(
                f"{stage_name}  no val improvement for {epochs_without_improvement} epochs, "
                f"lr -> {optimizer.param_groups[0]['lr']:.1e}",
                flush=True,
            )

    net.load_state_dict(best_state)
    torch.save(net.state_dict(), OUTPUT_PATH)
    print(
        f"{stage_name}  best val acc {best_acc:.3f}  saved {OUTPUT_PATH}",
        flush=True,
    )
    return best_acc


for param in model.parameters():
    param.requires_grad = False

opened = []
stage_accuracy = {}
for stage_name, modules in LAYER_STAGES:
    for module in modules:
        for param in module.parameters():
            param.requires_grad = True
    opened.extend(modules)
    trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
    print(f"\n=== {stage_name}  trainable params {trainable:,} ===", flush=True)
    stage_accuracy[stage_name] = train_stage(model, stage_name, opened)

print("stage best val acc:", {name: round(acc, 3) for name, acc in stage_accuracy.items()})


def per_class_accuracy(net):
    net.to(device)
    net.eval()
    correct = torch.zeros(num_classes)
    total = torch.zeros(num_classes)
    with torch.no_grad():
        for images, labels in val_loader:
            preds = net(images.to(device)).argmax(dim=1).cpu()
            for class_id in range(num_classes):
                chosen = labels == class_id
                total[class_id] += chosen.sum()
                correct[class_id] += (preds[chosen] == class_id).sum()
    accuracy = torch.zeros(num_classes)
    counted = total > 0
    accuracy[counted] = correct[counted] / total[counted]
    return accuracy.tolist()


class_accuracy = per_class_accuracy(model)
model.cpu()
if device.type == "mps":
    torch.mps.empty_cache()

painter_labels = [painters[i].replace("_", " ") for i in range(num_classes)]
fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(range(num_classes), class_accuracy, color="steelblue")
ax.set_yticks(range(num_classes))
ax.set_yticklabels(painter_labels)
ax.set_xlim(0, 1)
ax.set_xlabel("accuracy")
ax.set_title("resnet152 progressive fine-tune")
fig.tight_layout()
fig.savefig("per_class_accuracy_resnet152_progressive.png", dpi=150, bbox_inches="tight")
plt.show()
